In [ ]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    roc_auc_score,
)
from sklearn.impute import SimpleImputer


# Import embeddings and metadata

In [ ]:
embfile = '../Results/Emb-SIBIONICS-2022-06_1.csv'  # _meanpool
metafile = '../data/2022-06_1_meta.csv'
meta_id_col = 'id'

In [ ]:
embs = pd.read_csv(embfile,index_col=0)
embs.shape

In [ ]:
metafile = pd.read_csv(metafile)
metafile.index = metafile[meta_id_col]
metafile = metafile.loc[embs.index]
metafile.shape

# Prediction linear read-out
## Necessary functions

In [ ]:
from typing import Tuple, Optional, Dict, Any

def split_train_val_test(
    X: np.ndarray,
    y: np.ndarray,
    seed: int,
    test_size: float,
    val_size: float,
    stratify: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    val_size is a fraction of the total dataset (not of train+val).
    """
    if not (0.0 < test_size < 1.0) or not (0.0 < val_size < 1.0) or (test_size + val_size >= 1.0):
        raise ValueError("Require 0<test_size<1, 0<val_size<1, and test_size+val_size<1")

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=seed,
        stratify=stratify if stratify is not None else None
    )

    # val as fraction of total => within trainval, val_frac = val_size / (1 - test_size)
    val_frac = val_size / (1.0 - test_size)
    stratify_tv = None
    if stratify is not None:
        # need stratify for trainval subset
        # rebuild corresponding labels for trainval
        # easiest: compute using y_trainval if stratify==y
        stratify_tv = y_trainval

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=val_frac,
        random_state=seed,
        stratify=stratify_tv
    )
    return X_train, y_train, X_val, y_val, X_test, y_test

## Regression model from embeddings

In [ ]:
goal = 'Age'
metafileReg = metafile[metafile[goal].notna()]
metafileReg.shape

In [ ]:
embsReg = embs.loc[metafileReg.index]
embsReg.shape

In [ ]:
X = embsReg.to_numpy(dtype=float)
y = pd.to_numeric(metafileReg[goal], errors="coerce").to_numpy(dtype=float)
X_tr, y_tr, X_va, y_va, X_te, y_te = split_train_val_test(
            X, y,
            seed=123,
            test_size=0.2,
            val_size=0.1,
            stratify=None,
        )

In [ ]:
def build_reg_model(alpha: float = 1.0) -> Pipeline:
    # Simple + strong baseline: impute -> standardize -> ridge regression
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("model", Ridge(alpha=alpha, random_state=0)),
    ])

In [ ]:
model = build_reg_model(alpha=1.0)
model.fit(X_tr, y_tr)

In [ ]:
def pcc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if y_true.size < 2:
        return float("nan")
    # if constant, corr undefined
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])
def evaluate_reg(model: Pipeline, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    pred = model.predict(X)
    return {
        "PCC": pcc(y, pred),
        "MSE": float(mean_squared_error(y, pred)),
        "MAE": float(mean_absolute_error(y, pred)),
    }

In [ ]:
tr_m = evaluate_reg(model, X_tr, y_tr)
va_m = evaluate_reg(model, X_va, y_va)
te_m = evaluate_reg(model, X_te, y_te)

print("\n=== Regression ===")
print(f"Samples: total={len(y)} | train={len(y_tr)} | val={len(y_va)} | test={len(y_te)}")
print(f"Embedding dim: {X.shape[1]}")

def _fmt(m: Dict[str, float]) -> str:
    return f"PCC={m['PCC']:.4f}  MSE={m['MSE']:.4f}  MAE={m['MAE']:.4f}"

print("[Train] " + _fmt(tr_m))
print("[Val]   " + _fmt(va_m))
print("[Test]  " + _fmt(te_m))

## Classification model from embeddings

In [ ]:
goal = 'Type'
ExceptSet = ['Other','Special','GD']
metafileCls = metafile[(metafile[goal].notna())&(~metafile[goal].isin(ExceptSet))]
metafileCls.shape

In [ ]:
embsCls = embs.loc[metafileCls.index]
embsCls.shape

In [ ]:
y = metafileCls[goal].astype(str).to_numpy()
le = LabelEncoder()
y = le.fit_transform(y)
n_classes = len(le.classes_)
print(n_classes)

In [ ]:
X = embsCls.to_numpy(dtype=float)
X_tr, y_tr, X_va, y_va, X_te, y_te = split_train_val_test(
            X, y,
            seed=123,
            test_size=0.2,
            val_size=0.1,
            stratify=y,
        )

In [ ]:
from collections import Counter
def build_clf_model(C: float = 1.0, max_iter: int = 5000) -> Pipeline:
    # Baseline classifier: impute → standardize → logistic regression
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("model", LogisticRegression(
            C=C,
            max_iter=max_iter,
            solver="lbfgs",
            class_weight="balanced",
        )),
    ])
def evaluate_clf(model: Pipeline, X: np.ndarray, y: np.ndarray, n_classes: int) -> Dict[str, float]:
    pred = model.predict(X)
    acc = float(accuracy_score(y, pred))

    # AUC: requires probabilities
    auc = float("nan")
    try:
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)
            if n_classes == 2:
                # use positive class probability
                auc = float(roc_auc_score(y, proba[:, 1]))
            else:
                # multiclass macro OVR
                auc = float(roc_auc_score(y, proba, multi_class="ovr", average="macro"))
    except Exception:
        auc = float("nan")

    return {"AUC": auc, "ACC": acc}
def random_predict_with_prior(
    y_ref: np.ndarray,
    y_eval: np.ndarray,
    n_classes: int,
    seed: int = 0,
):
    """
    Random baseline that samples labels according to empirical class prior.

    y_ref  : labels used to estimate class prior (usually y_tr)
    y_eval : labels to evaluate against (y_tr / y_va / y_te)
    """
    rng = np.random.default_rng(seed)

    # empirical class distribution
    counts = Counter(y_ref)
    priors = np.array([counts.get(i, 0) for i in range(n_classes)], dtype=float)
    priors = priors / priors.sum()

    # random predictions
    y_rand = rng.choice(
        np.arange(n_classes),
        size=len(y_eval),
        p=priors
    )

    acc = accuracy_score(y_eval, y_rand)

    # random probabilities consistent with prior
    proba = np.tile(priors, (len(y_eval), 1))

    try:
        if n_classes == 2:
            auc = roc_auc_score(y_eval, proba[:, 1])
        else:
            auc = roc_auc_score(
                y_eval,
                proba,
                multi_class="ovr",
                average="macro"
            )
    except Exception:
        auc = float("nan")

    return {
        "ACC": acc,
        "AUC": auc,
        "priors": priors
    }
def fmt(m):
    auc = "nan" if np.isnan(m["AUC"]) else f"{m['AUC']:.4f}"
    return f"AUC={auc}  ACC={m['ACC']:.4f}"

In [ ]:
model = build_clf_model()
model.fit(X_tr, y_tr)

tr_m = evaluate_clf(model, X_tr, y_tr, n_classes=n_classes)
va_m = evaluate_clf(model, X_va, y_va, n_classes=n_classes)
te_m = evaluate_clf(model, X_te, y_te, n_classes=n_classes)
rand_tr = random_predict_with_prior(y_tr, y_tr, n_classes, seed=0)
rand_va = random_predict_with_prior(y_tr, y_va, n_classes, seed=1)
rand_te = random_predict_with_prior(y_tr, y_te, n_classes, seed=2)

print("\n=== Classification (Model vs Random Prior Baseline) ===")
print(f"Classes ({n_classes}): {list(le.classes_)}")
print(f"Embedding dim: {X.shape[1]}\n")

print("[Train] Model :", fmt(tr_m))
print("        Random:", fmt(rand_tr), "\n")

print("[Val]   Model :", fmt(va_m))
print("        Random:", fmt(rand_va), "\n")

print("[Test]  Model :", fmt(te_m))
print("        Random:", fmt(rand_te))


## Prediction in Multicenter data

In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.stats as stats
import sklearn

In [ ]:
emb = pd.read_csv('../Results/Emb-Multicenter_cont.csv',index_col=0)  # Emb-Multicenter_meanpool AllCGMMetric_Shanghai  vec_total_823_mergevec  Emb-Multicenter_cont
summary = pd.read_csv('../data/Summary_Shanghai.csv',index_col=0)
print(emb.shape, summary.shape)

In [ ]:
summary = summary.loc[emb.index]

In [ ]:
# for metrics
emb = (emb-emb.mean())/emb.std()

In [ ]:
summary.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import umap

In [ ]:
reducer = umap.UMAP(
    n_neighbors=100,
    min_dist=0.1,
    n_components=2,
    metric="euclidean",
    random_state=42,
)
umap_xy = reducer.fit_transform(emb.values)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
umap_xy = pca.fit_transform(emb.values)

In [ ]:
umap_df = pd.DataFrame(
    umap_xy,
    columns=["UMAP1", "UMAP2"],
    index=emb.index
)

umap_df["Age"] = summary["age"].values
umap_df["Type"] = summary["type"].values

In [ ]:
plt.figure(figsize=(6, 5))

sc = plt.scatter(
    umap_df["UMAP1"],
    umap_df["UMAP2"],
    c=umap_df["Age"],
    cmap="viridis",
    s=6,
    alpha=0.5
)

plt.colorbar(sc, label="Age")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP of Embeddings colored by Age")

plt.tight_layout()
#plt.savefig("../Results/UMAP_Multicenter_meanpool_age.png")
plt.show()

In [ ]:
keep_types = ['NGT', 'IGR', 'T2D']

plot_df = umap_df[umap_df["Type"].isin(keep_types)].copy()
type_palette = {
    'NGT':    'Green',   # gray
    'IGR': 'Orange',   # light blue
    'T2D':   'Red',   # red
}
plt.figure(figsize=(6, 5))

sns.scatterplot(
    data=plot_df,
    x="UMAP1",
    y="UMAP2",
    hue="Type",
    hue_order=['NGT', 'IGR', 'T2D'],
    palette=type_palette,
    s=10,
    alpha=0.5,
    linewidth=0
)

plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.title("PCA of Embeddings colored by Type")

plt.legend(
    title="Type",
    markerscale=2,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
#plt.savefig("../Results/UMAP_cons_type.png", dpi=300)
plt.show()


In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    mean_squared_error,
    r2_score
)


In [ ]:
X = emb.values  # (n, 128)

# ---- classification label ----
y_cls_raw = summary['type']  # NGT / IGR / T2D

# drop samples without label
mask_cls = y_cls_raw.notna()
X_cls = X[mask_cls.values]
y_cls_raw = y_cls_raw.loc[mask_cls]

le = LabelEncoder()
y_cls = le.fit_transform(y_cls_raw)

print("Classes:", dict(zip(le.classes_, range(len(le.classes_)))))

In [ ]:
pipe_cls = Pipeline([
    #('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=1000,
    ))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, aucs = [], []

for fold, (tr, te) in enumerate(skf.split(X_cls, y_cls), 1):
    pipe_cls.fit(X_cls[tr], y_cls[tr])

    y_pred = pipe_cls.predict(X_cls[te])
    y_prob = pipe_cls.predict_proba(X_cls[te])

    acc = accuracy_score(y_cls[te], y_pred)
    auc = roc_auc_score(
        y_cls[te],
        y_prob,
        multi_class='ovr',
        average='macro'
    )

    accs.append(acc)
    aucs.append(auc)

    print(f"[Fold {fold}] ACC={acc:.3f}  Macro-AUC={auc:.3f}")

print(f"\nOverall ACC = {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Overall AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")


In [ ]:
def eval_regression(X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    pccs, rmses, r2s = [], [], []

    for tr, te in kf.split(X):
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('reg', Ridge(alpha=1.0))
        ])
        pipe.fit(X[tr], y[tr])

        y_hat = pipe.predict(X[te])

        pccs.append(np.corrcoef(y[te], y_hat)[0, 1])
        rmses.append(np.sqrt(mean_squared_error(y[te], y_hat)))
        r2s.append(r2_score(y[te], y_hat))

    return (
        np.mean(pccs), np.std(pccs),
        np.mean(rmses), np.std(rmses),
        np.mean(r2s), np.std(r2s)
    )


In [ ]:
regression_keys = [
    'age',
    'bmi',
    'hba1c',
    'fpg',
    'hdl'
]

for key in regression_keys:
    if key not in summary.columns:
        continue

    y_raw = summary[key]
    mask = y_raw.notna()

    if mask.sum() < 50:
        print(f"{key:10s}: skipped (too few samples)")
        continue

    X_reg = X[mask.values]
    y_reg = y_raw.loc[mask].values.astype(float)

    pcc_m, pcc_s, rmse_m, rmse_s, r2_m, r2_s = eval_regression(X_reg, y_reg)

    print(
        f"{key:10s} | "
        f"PCC={pcc_m:.3f}±{pcc_s:.3f}  "
        f"RMSE={rmse_m:.3f}±{rmse_s:.3f}  "
        f"R2={r2_m:.3f}±{r2_s:.3f}"
    )


## Prediction in Zhao

In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.stats as stats
import sklearn
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge,ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
emb = pd.read_csv('../Results/zhao_CGMformer_mergevec.csv',index_col=0)  # Emb-Zhao_cont zhao_CGMformer_mergevec Emb-Zhao_meanpool
summary = pd.read_csv('../data/Summary_Zhao.csv', index_col=0)
label = pd.read_csv('../data/Zhao_label.csv', index_col=0)
print(emb.shape, summary.shape,label.shape)


In [ ]:
summary = summary.loc[emb.index]
summary.columns

In [ ]:
label = label.loc[emb.index]
label.columns

In [ ]:
summary[['Macrovascular', 'Microvascular','Complication']]=label[['Macrovascular', 'Microvascular','Complication']]
summary.columns

In [ ]:
X = emb.values  # (n, 128)

# ---- classification label ----
y_cls_raw = summary['Complication']  # NGT / IGR / T2D

# drop samples without label
mask_cls = y_cls_raw.notna()
X_cls = X[mask_cls.values]
y_cls_raw = y_cls_raw.loc[mask_cls]

le = LabelEncoder()
y_cls = le.fit_transform(y_cls_raw)

print("Classes:", dict(zip(le.classes_, range(len(le.classes_)))))
print(y_cls)

In [ ]:
pipe_cls = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        n_jobs=-1
    ))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

accs, aucs = [], []

for fold, (tr, te) in enumerate(skf.split(X_cls, y_cls), 1):
    pipe_cls.fit(X_cls[tr], y_cls[tr])

    y_pred = pipe_cls.predict(X_cls[te])
    y_prob = pipe_cls.predict_proba(X_cls[te])[:, 1]
    #print(y_cls[te])
    #print(y_pred)
    acc = accuracy_score(y_cls[te], y_pred)
    auc = roc_auc_score(y_cls[te], y_prob)

    accs.append(acc)
    aucs.append(auc)

    print(f"[Fold {fold}] ACC={acc:.3f}  Macro-AUC={auc:.3f}")

print(f"\nOverall ACC = {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"Overall AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")


In [ ]:
def eval_regression(X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    pccs, rmses, r2s = [], [], []

    for tr, te in kf.split(X):
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('reg', ElasticNet(
                alpha=0.01,
                l1_ratio=0.5,
                max_iter=5000
            ))
        ])
        pipe.fit(X[tr], y[tr])

        y_hat = pipe.predict(X[te])
        print(
            np.std(y[te]),
            np.std(y_hat)
        )
        pccs.append(np.corrcoef(y[te], y_hat)[0, 1])
        rmses.append(np.sqrt(mean_squared_error(y[te], y_hat)))
        r2s.append(r2_score(y[te], y_hat))
        #print(pccs,rmses,r2s)
    return (
        np.mean(pccs), np.std(pccs),
        np.mean(rmses), np.std(rmses),
        np.mean(r2s), np.std(r2s)
    )

In [ ]:
regression_keys = [
    'age',
    'bmi',
    'hba1c',
    'fpg',
    'hdl'
]
X = emb.values  # (n, 128)
for key in regression_keys:
    if key not in summary.columns:
        continue

    y_raw = summary[key]
    mask = y_raw.notna()
    if mask.sum() < 50:
        print(f"{key:10s}: skipped (too few samples)")
        continue
    X_reg = X[mask.values]
    y_reg = y_raw.loc[mask].values.astype(float)
    pcc_m, pcc_s, rmse_m, rmse_s, r2_m, r2_s = eval_regression(X_reg, y_reg)
    print(
        f"{key:10s} | "
        f"PCC={pcc_m:.3f}±{pcc_s:.3f}  "
        f"RMSE={rmse_m:.3f}±{rmse_s:.3f}  "
        f"R2={r2_m:.3f}±{r2_s:.3f}"
    )

# UMAP for embeddings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import umap


In [ ]:
embfile = '../Results/Emb-SIBIONICS-2022-06_1_cont.csv'  #  Emb-SIBIONICS-2022-06_1_meanpool Emb-SIBIONICS-2022-06_1_cont
metafile = '../data/2022-06_1_meta.csv'
meta_id_col = 'id'

In [ ]:
embs = pd.read_csv(embfile, index_col=0)
metafile = pd.read_csv(metafile)
metafile.index = metafile[meta_id_col]
metafile = metafile.loc[embs.index]
metafile.shape

In [ ]:
np.random.seed(42)
assert embs.shape[0] == metafile.shape[0]
assert embs.index.equals(metafile.index)

print("Embedding dim:", embs.shape[1])
print("Age NaNs:", metafile["Age"].isna().sum())
print("Type NaNs:", metafile["Type"].isna().sum())


In [ ]:
reducer = umap.UMAP(
    n_neighbors=100,
    min_dist=0.1,
    n_components=2,
    metric="euclidean",
    random_state=42,
)

umap_xy = reducer.fit_transform(embs.values)


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
umap_xy = pca.fit_transform(embs.values)

In [ ]:
umap_df = pd.DataFrame(
    umap_xy,
    columns=["UMAP1", "UMAP2"],
    index=embs.index
)

umap_df["Age"] = metafile["Age"].values
umap_df["Type"] = metafile["Type"].values


In [ ]:
plt.figure(figsize=(6, 5))

sc = plt.scatter(
    umap_df["UMAP1"],
    umap_df["UMAP2"],
    c=umap_df["Age"],
    cmap="viridis",
    s=6,
    alpha=0.5
)

plt.colorbar(sc, label="Age")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP of Embeddings colored by Age")

plt.tight_layout()
#plt.savefig("../Results/UMAP_mean_age.png")
plt.show()


In [ ]:
keep_types = ['ND', 'HRT1D', 'HRT2D', 'T1D', 'T2D']

plot_df = umap_df[umap_df["Type"].isin(keep_types)].copy()
type_palette = {
    'ND':    '#d6d627',   # gray
    'T1D':   '#1f77b4',   # blue
    'HRT1D': '#aec7e8',   # light blue
    'T2D':   '#d62728',   # red
    'HRT2D': '#ff9896',   # light red
}

In [ ]:
plt.figure(figsize=(6, 5))

sns.scatterplot(
    data=plot_df,
    x="UMAP1",
    y="UMAP2",
    hue="Type",
    hue_order=['ND', 'T1D', 'HRT1D', 'T2D', 'HRT2D'],
    palette=type_palette,
    s=10,
    alpha=0.5,
    linewidth=0
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP of Embeddings colored by Type")

plt.legend(
    title="Type",
    markerscale=2,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
#plt.savefig("../Results/UMAP_cons_type.png", dpi=300)
plt.show()


In [ ]:
# size of each island by connected component
from sklearn.neighbors import NearestNeighbors
import networkx as nx

nbrs = NearestNeighbors(n_neighbors=15).fit(embs.values)
knn = nbrs.kneighbors_graph(mode="connectivity")

G = nx.from_scipy_sparse_array(knn)
components = list(nx.connected_components(G))
[len(c) for c in sorted(components, key=len, reverse=True)]


# PCA for embeddings

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# standardize first (important)
X = StandardScaler().fit_transform(embs.values)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=embs.index
)

pca_df["Age"] = metafile["Age"].values
pca_df["Type"] = metafile["Type"].values


In [ ]:
keep_types = ['ND', 'HRT1D', 'HRT2D', 'T1D', 'T2D']
plot_df = pca_df[pca_df["Type"].isin(keep_types)].copy()

type_palette = {
    'ND':    '#d6d627',
    'T1D':   '#1f77b4',
    'HRT1D': '#aec7e8',
    'T2D':   '#d62728',
    'HRT2D': '#ff9896',
}

plt.figure(figsize=(6, 5))

sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="Type",
    hue_order=['ND', 'T1D', 'HRT1D', 'T2D', 'HRT2D'],
    palette=type_palette,
    s=10,
    alpha=0.8,
    linewidth=0
)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("PCA of Embeddings colored by Type")

plt.legend(frameon=False, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig("../Results/PCA_cons_type.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))

sc = plt.scatter(
    pca_df["PC1"],
    pca_df["PC2"],
    c=pca_df["Age"],
    cmap="viridis",
    s=6,
    alpha=0.8
)

plt.colorbar(sc, label="Age")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Embeddings colored by Age")

plt.tight_layout()
plt.savefig("../Results/PCA_cons_age.png", dpi=300)

plt.show()


In [ ]:
plt.figure(figsize=(6, 4))

for t, c in type_palette.items():
    vals = plot_df.loc[plot_df["Type"] == t, "PC1"]
    sns.kdeplot(vals, label=t, color=c, linewidth=2)

plt.xlabel("PC1")
plt.title("PC1 density by Type")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("../Results/PCA_PC1_mean_type.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))

for t, c in type_palette.items():
    vals = plot_df.loc[plot_df["Type"] == t, "PC2"]
    sns.kdeplot(vals, label=t, color=c, linewidth=2)

plt.xlabel("PC2")
plt.title("PC2 density by Type")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("../Results/PCA_PC2_mean_type.png", dpi=300)
plt.show()

# Analysis For the DNB calculation

In [ ]:
metricspth = '../Results/dnb_metrics.csv'
datapth = '../data/2022-06_1.csv'
metafile = '../data/2022-06_1_meta.csv'
meta_id_col = 'id'

In [ ]:
metrics = pd.read_csv(metricspth,index_col=0)
metrics = metrics[metrics['num_patches']>200]
metrics.shape

In [ ]:
data = pd.read_csv(datapth,index_col=0)
data = data.loc[metrics.index]
data.shape

In [ ]:
metainfo = pd.read_csv(metafile,index_col=0)
metainfo.index = metainfo[meta_id_col]
metainfo = metainfo.loc[metrics.index]
metainfo.shape

In [ ]:
data = data[[str(x) for x in range(3456)]]
metrics['Mean']=data.mean(axis=1)

In [ ]:
metrics['TIR']=(((data<=10)&(data>=3.9)).sum(axis=1))/((data.notna()).sum(axis=1))

In [ ]:
metrics['SD']=data.std(axis=1)

In [ ]:
metrics['Age'] = metainfo['Age']

In [ ]:
import seaborn as sns

In [ ]:
metrics = metrics[metrics['Age'].notna()]
metrics.shape

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select numeric columns only
corr_cols = [
    'dnb_sd', 'dnb_pcc', 'lyap_proxy_log_smaxA',
    'Mean', 'TIR', 'SD', 'Age'
]

corr = metrics[corr_cols].corr(method='spearman')  # Spearman is safer

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    cbar_kws={'label': 'Spearman ρ'}
)
plt.title("Correlation among CGM and dynamical metrics")
plt.tight_layout()
plt.savefig('../Results/Correlation.png')
plt.show()


In [ ]:
metrics['Label'] = metainfo.loc[metainfo.index, 'Type']
df_bin = metrics[metrics['Label'].isin(['ND', 'T2D', 'HRT2D'])].copy()

label_map = {
    'ND': 0,
    'T2D': 1,
    'HRT2D': 2
}
df_bin['y'] = df_bin['Label'].map(label_map)

print(df_bin['Label'].value_counts())

In [ ]:
feature_groups = {
    "CGM_only": ['Mean', 'TIR', 'SD'],
    "Dynamical_only": ['dnb_sd', 'dnb_pcc', 'lyap_proxy_log_smaxA'],
    "CGM_plus_Dynamical": [
        'Mean', 'TIR', 'SD',
        'dnb_sd', 'dnb_pcc', 'lyap_proxy_log_smaxA'
    ],
}


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, make_scorer
import numpy as np


In [ ]:
def eval_multiclass_auc(df, features, y_col='y', n_splits=5):
    X = df[features + ['Age']].values
    y = df[y_col].values

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
                solver='lbfgs',
                max_iter=3000,
                class_weight='balanced'
            )
        )
    ])

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []

    for train_idx, test_idx in cv.split(X, y):
        pipe.fit(X[train_idx], y[train_idx])
        prob = pipe.predict_proba(X[test_idx])

        auc = roc_auc_score(
            y[test_idx],
            prob,
            multi_class='ovr',
            average='macro'
        )
        aucs.append(auc)

    return np.mean(aucs), np.std(aucs)


In [ ]:
for name, feats in feature_groups.items():
    mean_auc, std_auc = eval_multiclass_auc(df_bin, feats)
    print(f"{name:25s}  Macro-AUC = {mean_auc:.3f} ± {std_auc:.3f}")


## DNB in Colas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

In [ ]:
dnb = pd.read_csv('../Results/Colas_dnb_metrics.csv',index_col=0)
rawdnb = pd.read_csv('../Results/ColasDNB.csv',index_col=0)
summary = pd.read_csv('../data/Summary_Colas.csv',index_col=0)
data = pd.read_csv('../data/Colas.csv',index_col=0)
print(dnb.shape,summary.shape)

In [ ]:
cgmmetric = pd.DataFrame(index=data.index)
cgmmetric['mean'] = data.mean(axis=1)
cgmmetric['TIR'] = ((data>=3.9)&(data<=10)).sum(axis=1)/(data.notna().sum(axis=1))
cgmmetric['SD'] = data.std(axis=1)

In [ ]:
idmap = dict(zip(summary['Rawid'], summary.index))
rawdnb.index = [idmap[x] for x in rawdnb.index]

In [ ]:
summary = summary.loc[dnb.index]
summary.columns

In [ ]:
summary = summary[(summary.index.isin(dnb.index))&(summary.index.isin(rawdnb.index))]
rawdnb = rawdnb.loc[summary.index]
dnb = dnb.loc[summary.index]
rawdnb.columns

In [ ]:
month = 24
summaryfilter = summary[(summary['followup']>30*month)&(summary['type']=='NGT')|(summary['followup']<30*month)&(summary['type']=='T2D')]
print(summaryfilter.shape)
summaryfilter = summaryfilter[summaryfilter['hba1c'].notna()]
print(summaryfilter.shape)

In [ ]:
dnbfilter = dnb.loc[summaryfilter.index]
rawdnbfilter = rawdnb.loc[summaryfilter.index]
cgmmetricfilter = cgmmetric.loc[summaryfilter.index]

In [ ]:
y = summaryfilter['type'].apply(lambda x:{'NGT':0,'T2D':1}[x])
print(sum(y),len(y))

In [ ]:
results = dict()
#results['DNB_PCC']=roc_auc_score(y,-dnbfilter['dnb_pcc'])
results['DNB_SD']=roc_auc_score(y,dnbfilter['dnb_sd'])
#results['RawDNB_PCC']=roc_auc_score(y,rawdnbfilter['pcc'])
results['RawDNB_SD']=roc_auc_score(y,rawdnbfilter['sd'])
results['HbA1c']=roc_auc_score(y,summaryfilter['hba1c'])
results['FPG']=roc_auc_score(y,summaryfilter['fpg'])
#results['mean']=roc_auc_score(y,cgmmetricfilter['mean'])
#results['TIR']=roc_auc_score(y,-cgmmetricfilter['TIR'])
results['SD']=roc_auc_score(y,cgmmetricfilter['SD'])

In [ ]:
plt.figure(figsize=(5,5))
plt.bar(results.keys(),results.values())
plt.ylim([0.5,1])
plt.ylabel('AUC')
plt.title(f'sample follow up {month} months')
plt.show()

In [ ]:
plt.scatter(cgmmetricfilter['mean'],dnbfilter['dnb_sd'],c=[['green','red'][x] for x in y])
plt.show()

In [ ]:
dnbfilter

In [ ]:
summaryfilter.loc['Colas_NGT_case  178']

In [ ]:
plt.scatter(dnbfilter['dnb_pcc'],dnbfilter['dnb_sd'],c=[['green','red'][x] for x in y])
plt.show()